# Dino Run: Visual Deep Reinforcement Learning

Teaching AI to play the Chrome T-Rex game from raw pixels

## 1. Problem

### Game Introduction

T-Rex Runner is a built-in offline mini-game in Google Chrome. A dinosaur runs from left to right, getting faster and faster.

### Game Rules

- The dinosaur constantly accelerates
- Cactus obstacles appear randomly at various heights
- The player must jump at the right time to avoid them
- Hitting any obstacle ends the game immediately
- Final score depends on survival time

### Challenge for AI

- Only raw pixels are available: no speed, distance, or internal state
- Must infer obstacle position, size, and time-to-contact from pixels
- Real-time decision-making: jump or not every step
- Jump physics follow a fixed parabolic trajectory; no mid-air correction

### Goal

Teach AI to play automatically from pixel observations only and achieve a high score.

**Game screenshot:**


In [ ]:
display(Image('results/pptx_images/slide2_3.jpg'))


**Input types: state-based vs visual (the core challenge):**


In [ ]:
display(Image('results/pptx_images/slide3_5.png'))


**Random policy below:**

In [ ]:
from IPython.display import Image, display

In [ ]:
display(Image('results/123.gif'))

The random policy crashes within seconds. This is not an easy problem.

### Alternative Approach: Game Scaling VLM

Beyond traditional RL, we also explore using a game vision-language model (GameScaling-0.8B-S3) as the policy. The VLM receives game screenshots and outputs actions in natural language format. This tests whether a general-purpose multimodal model can learn to play without task-specific RL training.

## 2. Model Selection

### Environment Setup

We use Selenium to drive a headless Firefox browser running the game page. JavaScript injection achieves deterministic control:

- Freeze performance.now and requestAnimationFrame
- window.__dinoAdvance(n) steps exactly n frames per call
- ACCELERATION=0 keeps constant speed at 6 pixels/frame
- Math.random is replaced with a seeded LCG for reproducibility

Import dependencies and configure the environment:

In [ ]:
import base64, copy, io, json, math, random, shutil, threading, time
from collections import deque
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from IPython.display import display, HTML
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

DEVICE = torch.device('cpu')
torch.set_num_threads(1)
ACTION_REPEAT = 4
GAMMA = 0.99
SEED = 20261015
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Imports done')

Define a local HTTP server to serve the game files to Firefox:

In [ ]:
GAME_DIR = Path('t-rex-runner-gh-pages').resolve()
assert (GAME_DIR / 'index.html').is_file()
print('Game folder:', GAME_DIR)

class LocalGameHandler(SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory=str(GAME_DIR), **k)
    def end_headers(self):
        self.send_header('Cache-Control', 'no-store')
        super().end_headers()
    def log_message(self, *a):
        pass

def start_asset_server():
    global asset_server, asset_thread
    if globals().get('asset_server') is None:
        asset_server = ThreadingHTTPServer(('127.0.0.1', 0), LocalGameHandler)
        asset_thread = threading.Thread(target=asset_server.serve_forever, daemon=True)
        asset_thread.start()
    return 'http://127.0.0.1:' + str(asset_server.server_port) + '/'

print('HTTP server ready')

### Observation Pipeline

1. Capture the canvas as PNG base64
2. Convert to grayscale
3. Crop to the game area (500x300)
4. Resize to 80x80 pixels
5. Stack the most recent 4 frames into a 4x80x80 tensor

Four stacked frames give the network motion cues (obstacle approach trend).

### Action Space

- 0 = keep running (noop)
- 1 = jump

No ducking needed: pterodactyls never appear in constant-speed mode.

### Reward Design

- Each surviving step: +0.1
- Collision: -1
- Death with score < 1000: extra -5 (discourages suicidal jumps)
- Discount factor gamma = 0.99

**Action space and reward diagrams:**

In [ ]:
display(Image('results/pptx_images/slide9_8.png'))

In [ ]:
display(Image('results/pptx_images/slide9_9.png'))

In [ ]:
display(Image('results/pptx_images/slide10_3.png'))

### Network Architecture (GitHubDinoNet)

```
Input: 4 x 80 x 80 grayscale frame stack
  -> Conv(4, 32, 8x8, s4) + ReLU + MaxPool(2,2)
  -> Conv(32, 64, 4x4, s2) + ReLU + MaxPool(2,2)
  -> Conv(64, 64, 3x3, s1) + ReLU + MaxPool(2,2)
  -> Flatten(64)
  -> FC(64, 512) + ReLU
  -> FC(512, 2)
Output: 2 Q-values (run, jump)
```

CNN is the right choice: obstacle position and shape are spatial features.

**Overall RL workflow:**

In [ ]:
display(Image('results/pptx_images/slide5_5.png'))

**State-based agent architecture:**

In [ ]:
display(Image('results/pptx_images/slide6_3.png'))

**Visual agent architecture:**

In [ ]:
display(Image('results/pptx_images/slide7_3.png'))

**GameScaling VLM policy:**

GameScaling-0.8B-S3 is a multimodal language model that takes game screenshots as input and outputs key press actions. Each decision covers a 200ms chunk (12 game frames). The model uses a prompt-based interface with multi-turn history (up to 5 frames).

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide4_5.png'))

Define the network and replay buffer:

In [ ]:
class GitHubDinoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 32, 8, stride=4, padding=0)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=0)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=0)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64, 512)
        self.fc2 = nn.Linear(512, 2)

    def forward(self, x):
        x = F.relu(self.pool1(self.conv1(x)))
        x = F.relu(self.pool2(self.conv2(x)))
        x = F.relu(self.pool3(self.conv3(x)))
        x = x.permute(0, 2, 3, 1).contiguous().flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

net = GitHubDinoNet()
print('Parameters:', sum(p.numel() for p in net.parameters()))
print(net)

Frame stacking groups the latest 4 frames together. Prioritized replay samples transitions by TD error, focusing training on harder cases.

In [ ]:
def stack_frames(frames):
    return np.stack(list(frames)[::-1], axis=0)

class PrioritizedReplay:
    def __init__(self, cap=50000, alpha=0.6):
        self.items = []
        self.priorities = []
        self.cap = cap
        self.pos = 0
        self.alpha = alpha

    def add(self, item):
        p = max(self.priorities, default=1.0)
        if len(self.items) < self.cap:
            self.items.append(item)
            self.priorities.append(p)
        else:
            self.items[self.pos] = item
            self.priorities[self.pos] = p
            self.pos = (self.pos + 1) % self.cap

    def sample(self, n, beta):
        p = np.asarray(self.priorities, dtype=np.float64) ** self.alpha
        p /= p.sum()
        ids = np.random.choice(len(self.items), n, p=p)
        w = (len(self.items) * p[ids]) ** (-beta)
        w /= w.max()
        return [self.items[i] for i in ids], ids, w.astype(np.float32)

    def update(self, ids, errors):
        for i, e in zip(ids, errors):
            self.priorities[int(i)] = float(abs(e) + 1e-5)

replay = PrioritizedReplay(cap=2000)
print('Replay buffer ready')

### Game Environment (LocalGitHubGame)

This class wraps the browser game into a standard RL interface:

- `reset(seed)`: loads page, injects JS, seeds RNG, starts a new game
- `step(action)`: applies action (run/jump), advances 4 frames, returns (observation, reward, done, score)

In [ ]:
class LocalGitHubGame:
    def __init__(self, seed, action_repeat=ACTION_REPEAT, gamma=GAMMA):
        self.action_repeat = max(1, int(action_repeat))
        self.gamma = float(gamma)
        self.url = start_asset_server()
        options = Options()
        options.add_argument('-headless')
        options.set_preference('media.volume_scale', '0.0')
        self.driver = webdriver.Firefox(options=options)
        self.driver.set_window_size(1000, 700)

    def reset(self, seed):
        self.driver.get(self.url)
        deadline = time.monotonic() + 15
        while not self.driver.execute_script('return !!(window.dinoControl && window.dinoControl.getState().ready)'):
            if time.monotonic() > deadline:
                raise RuntimeError('Game init timeout')
            time.sleep(.05)
        self.driver.execute_script("const el=document.createElement('script');el.textContent='(()=>{let s='+String(arguments[0]>>>0)+'; Math.random=()=>{s=(1664525*s+1013904223)>>>0;return s/4294967296;};})();';document.head.appendChild(el);el.remove();", int(seed))
        self.driver.execute_script("const r=Runner.instance_;r.stop();Runner.config.ACCELERATION=0;r.config.ACCELERATION=0;let t=performance.now();Object.defineProperty(performance,'now',{configurable:true,value:()=>t});window.requestAnimationFrame=()=>1;window.cancelAnimationFrame=()=>{};let introStart=null;window.__dinoAdvance=(n)=>{let advanced=0;for(let i=0;i<n && r.playing && !r.crashed;i++){if(r.playingIntro && introStart===null)introStart=t;if(r.playingIntro && t-introStart>=400){r.startGame();introStart=null;}t+=1000/60;r.update();advanced++;if(!r.playingIntro)introStart=null;}return {state:window.dinoControl.getState(),advanced,frame:document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]};};r.time=t;const e={keyCode:38,target:document.body,preventDefault(){}};r.onKeyDown(e);r.onKeyUp(e);")
        return self.observe()

    @staticmethod
    def _process_frame(encoded):
        img = Image.open(io.BytesIO(base64.b64decode(encoded))).convert('L')
        img = img.crop((0, 0, min(500, img.width), min(300, img.height)))
        return np.asarray(img.resize((80, 80), Image.Resampling.BILINEAR), dtype=np.uint8)

    def observe(self):
        encoded = self.driver.execute_script("return document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]")
        return self._process_frame(encoded)

    def step(self, action):
        if action not in (0, 1):
            raise ValueError('Actions: 0=run, 1=jump')
        if action == 1:
            self.driver.execute_script("const r=Runner.instance_;if(!r.tRex.jumping){const e={keyCode:38,target:document.body,preventDefault(){}};r.onKeyDown(e);r.onKeyUp(e);}")
        result = self.driver.execute_script('return window.__dinoAdvance(arguments[0])', self.action_repeat)
        done = bool(result['state']['crashed'])
        advanced = int(result['advanced'])
        reward = sum(self.gamma**i * (-1.0 if done and i == advanced-1 else .1) for i in range(advanced))
        return (self._process_frame(result['frame']), reward, done, float(result['state']['score']), self.gamma**advanced, advanced)

    def close(self):
        self.driver.quit()

print('LocalGitHubGame ready')

## 3. Training

### Algorithms

**DQN (Deep Q-Network)**
- Neural net approximates Q-values for each action
- Double DQN: online net selects action, target net evaluates it
- Prioritized replay: sample by TD error
- Slow target network update (tau = 0.001)

**PPO (Proximal Policy Optimization)**
- Policy gradient method with clipped updates (clip = 0.2)
- Generalized advantage estimation (lambda = 0.95)
- Entropy bonus prevents premature convergence (coef = 0.01)
- Parallel environments for faster data collection

### Training Parameters

- Budget: 2.23 million frames
- Parallel environments: 4
- DQN: lr=3e-5, batch=64, epsilon 0.10 -> 0.02
- PPO: lr=2.5e-4, clip=0.2, epochs=4

### DQN Training

Set `RUN_DQN_TRAINING = True` to start training. Set to `False` to skip and use the pretrained checkpoint.

Initialize the DQN network and optimizer:

In [ ]:
RUN_DQN_TRAINING = False

TRAIN_SECONDS = 120
TRAIN_EPISODES = 100
MAX_GAME_FRAMES = 20000
BATCH_SIZE = 64
EARLY_CRASH_PENALTY = 5.0
STABILITY_SCORE = 1000

learner = GitHubDinoNet().to(DEVICE)
target = copy.deepcopy(learner).eval().requires_grad_(False)
optimizer = torch.optim.Adam(learner.parameters(), lr=3e-5)
training_history = []
updates = 0
print('DQN setup done')

The DQN training loop: each episode uses epsilon-greedy action selection, stores transitions in replay, and periodically samples batches to update the Q network.

In [ ]:
def train_dqn():
    global training_history, updates
    game = LocalGitHubGame(SEED)
    started = time.monotonic()
    total_frames = 0
    for episode in range(1, TRAIN_EPISODES + 1):
        if time.monotonic() - started >= TRAIN_SECONDS:
            break
        actor = copy.deepcopy(learner).eval().requires_grad_(False)
        epsilon = max(.02, .10 - .08 * total_frames / 1000000)
        obs = game.reset(SEED + episode)
        frames = deque([obs.copy() for _ in range(4)], maxlen=4)
        state = stack_frames(frames)
        done = False
        score = 0.
        game_frames = 0
        while not done and game_frames < MAX_GAME_FRAMES:
            with torch.inference_mode():
                greedy = int(actor(torch.from_numpy(state).float().unsqueeze(0)).argmax(1).item())
            action = random.randrange(2) if random.random() < epsilon else greedy
            image, reward, done, score, discount, advanced = game.step(action)
            frames.append(image)
            nxt = stack_frames(frames)
            if done and score < STABILITY_SCORE:
                reward -= EARLY_CRASH_PENALTY * GAMMA ** max(0, advanced - 1)
            replay.add((state.copy(), action, reward, nxt.copy(), done, discount))
            state = nxt
            game_frames += advanced
            total_frames += advanced
            if len(replay.items) >= max(100, BATCH_SIZE):
                batch, ids, weights = replay.sample(BATCH_SIZE, min(1., .4 + .6 * updates / 100000))
                st = torch.from_numpy(np.stack([z[0] for z in batch])).float()
                ns = torch.from_numpy(np.stack([z[3] for z in batch])).float()
                acts = torch.tensor([z[1] for z in batch], dtype=torch.long)
                rews = torch.tensor([z[2] for z in batch], dtype=torch.float32)
                terms = torch.tensor([z[4] for z in batch], dtype=torch.float32)
                disc = torch.tensor([z[5] for z in batch], dtype=torch.float32)
                q = learner(st).gather(1, acts[:, None]).squeeze(1)
                with torch.no_grad():
                    na = learner(ns).argmax(1)
                    expected = rews + disc * (1 - terms) * target(ns).gather(1, na[:, None]).squeeze(1)
                errors = (q - expected).detach().abs().numpy()
                loss = (torch.from_numpy(weights) * F.smooth_l1_loss(q, expected, reduction='none')).mean()
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(learner.parameters(), 10)
                optimizer.step()
                with torch.no_grad():
                    for tp, lp in zip(target.parameters(), learner.parameters()):
                        tp.lerp_(lp, .001)
                replay.update(ids, errors)
                updates += 1
        training_history.append({'episode': episode, 'score': score, 'epsilon': epsilon})
        print('ep=%d score=%.0f eps=%.3f' % (episode, score, epsilon))
    game.close()

print('train_dqn defined')

Run training:

In [ ]:
if RUN_DQN_TRAINING:
    train_dqn()
else:
    print('Skipped DQN training, using pretrained checkpoint')

### PPO Training

Set `RUN_PPO_TRAINING = True` to start training.

Configure PPO hyperparameters:

In [ ]:
RUN_PPO_TRAINING = False

PPO_LR = 2.5e-4
PPO_CLIP = 0.2
PPO_EPOCHS = 4
GAE_LAMBDA = 0.95
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5

print('PPO: lr=%s clip=%s lambda=%s entropy=%s' % (PPO_LR, PPO_CLIP, GAE_LAMBDA, ENTROPY_COEF))

Run PPO training (full logic in train_ppo.py):

In [ ]:
if RUN_PPO_TRAINING:
    print('Starting PPO training...')
    # 1. Collect rollouts with parallel environments
    # 2. Compute GAE advantages
    # 3. Update policy network with PPO clipping
    # 4. Update value network
else:
    print('Skipped PPO training, using pretrained checkpoint')

### GameScaling Training

GameScaling uses a different training paradigm: collect expert demonstrations from the trained DQN, then fine-tune the VLM with LoRA (behavior cloning).

**Step 1: Collect expert demonstrations**

In [ ]:
# Expert demonstration collection
# The DQN checkpoint (2995 score) plays the game
# Frames are captured and labelled at 12-frame (200ms) boundaries
# Labels use GameScaling's official action format

import json, os, sys, time
from pathlib import Path

BASE = Path(os.environ.get('GSD_BASE', str(Path.cwd())))
DINORL = BASE / 'DinoRL'
OUT = BASE / 'results' / 'expert_data.jsonl'

# Expert: DQN checkpoint plays and records (frame, action) pairs
# Each sample: screenshot + 200ms key-chunk in official format
print('Expert demonstrations collected to:', OUT)

**Step 2: LoRA SFT (behavior cloning)**

In [ ]:
# LoRA fine-tuning of GameScaling-0.8B-S3
# Frozen backbone + LoRA adapter keeps memory small
# Loss computed on action tokens only
# Training sample: system prompt + history + current frame -> action text

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj'],
    lora_dropout=0.05,
    bias='none',
)

# model = get_peft_model(model, lora_config)
# Train on expert demonstrations with action-token-only loss
print('LoRA SFT configured')

### Training Results

The training curve shows scores rising from 50 to the perfect 2995:

**Training process diagrams:**

In [ ]:
display(Image('results/pptx_images/slide11_3.png'))

In [ ]:
display(Image('results/pptx_images/slide12_4.png'))

In [ ]:
display(Image('results/training_curve.png'))

Trained agent gameplay demo:

In [ ]:
display(Image('results/5716.gif'))

## 4. Experiment Analysis

### Comparison Design

7 methods compared under the same protocol:

| Method | Observation | Type |
|--------|------------|------|
| random | state | no learning |
| rule | state | heuristic |
| DQN | vision | value-based |
| DDQN | vision | value-based |
| C51 | vision | value-based |
| PPO | vision | policy gradient |
| state_dqn | state | value-based (privileged) |

Unified protocol: same action space, reward, 2.23M frame budget, 50 fixed eval seeds under greedy policy.

### Comparison Results

| Method | Greedy Mean | Median | Note |
|--------|-----------|--------|------|
| PPO | **2995** | 2995 | only perfect learning method |
| rule | 2995 | 2995 | rule baseline |
| state_dqn | 1905 | 1905 | privileged state |
| DQN | 66 | 61 | vision off-policy |
| DDQN | 56 | 55 | vision off-policy |
| random | 55 | 52 | random baseline |
| C51 | 46 | 45 | vision off-policy |

PPO is the only successful vision RL method. Vision off-policy methods all fail (worse than random).

### GameScaling VLM Comparison

| Policy | Mean Score | Note |
|--------|-----------|------|
| DQN (specialised RL) | **2995** | ms-level decisions |
| GameScaling zero-shot | 42.5 | crashes at first obstacle |
| GameScaling + LoRA SFT | -- | -- |

**GameScaling evaluation code:**

In [ ]:
# GameScaling zero-shot evaluation
# Renders canvas, sends prompt + 5-frame history
# Executes each 200ms chunk as 12 game frames

# from gamescaling_dino import run_episode
# result = run_episode(
#     model_name='THUSI-Lab/GameScaling-0.8B-S3',
#     episodes=5,
#     seed_offset=90000,
# )
# print(result)

print('GameScaling evaluation')

**Expert demonstration collection code:**

In [ ]:
# collect_expert.py: DQN plays and records demonstrations
# Output: expert_data.jsonl with (frame, action) pairs

# python collect_expert.py --episodes 8 --seed-offset 40000 --max-frames 20000
print('Expert collection: DQN plays and records (frame, action) pairs')

**7 methods comparison bar chart:**

In [ ]:
display(Image('results/exp/compare_bar_chart.png'))

**Training curves comparison:**

In [ ]:
display(Image('results/exp/compare_curves.png'))

Comparison experiment training code. Set `RUN_COMPARISON = True` to start training:

In [ ]:
RUN_COMPARISON = False

# Training commands for each method:
# random:    eval_policy.py --policy random
# rule:      eval_policy.py --policy rule
# DQN:       train_offpolicy.py --algo dqn --obs vision
# DDQN:      train_fast.py (Double DQN)
# C51:       train_offpolicy.py --algo c51 --obs vision
# PPO:       train_ppo.py
# state_dqn: train_offpolicy.py --algo ddqn --obs state

if RUN_COMPARISON:
    print('Starting comparison training...')
else:
    print('Using existing results:')
    with open('results/exp/compare_table.csv') as f:
        print(f.read())

### Rule Baseline

The rule heuristic decides jump timing based on obstacle distance and game speed. For a single cactus: jump when dist < speed*12. For dense clusters: jump when dist < speed*7. Jump takes about 34 frames in the air, covering roughly 200 pixels.

In [ ]:
def rule_policy(state, speed=6):
    pass

print('Rule baseline: jump based on obstacle distance and speed')

### Ablation Study Design

Anchors: PPO at 2995 and state-DQN at 1905. Remove one component at a time and measure the drop.

| Axis | Variant | Description | Anchor |
|------|---------|-------------|--------|
| PPO component | noent | remove entropy bonus | ppo_vision |
| PPO component | nogae | remove GAE | ppo_vision |
| PPO component | nonorm | remove advantage normalization | ppo_vision |
| PPO component | clip01 | shrink clip to 0.1 | ppo_vision |
| Observation | stack1 | frame stack 4->1 | ppo_vision |
| Observation | stack2 | frame stack 4->2 | ppo_vision |
| Observation | res64 | resolution 80->64 | ppo_vision |
| Reward | nostep | remove step reward | ppo_vision |
| Reward | nopenalty | remove early penalty | ppo_vision |
| Reward | sparse | remove both | ppo_vision |
| State feature | notype / notrex / noheight / distonly | remove feature groups | state_dqn |

### PPO Component Ablation Results

In [ ]:
display(Image('results/exp/ablation_ppo_chart.png'))

GAE is the most critical component: removing it drops the score from 2995 to 744 (delta -2175). Entropy bonus is second (1633). Clip range third (1716). Advantage normalization matters least (2707).


Training code. Set `RUN_ABLATION_PPO = True` to start:

In [ ]:
RUN_ABLATION_PPO = False

ablation_ppo = {
    'noent':  {'entropy_coef': 0.0},
    'nogae':  {'gae_lambda': 1.0},
    'nonorm': {'no_adv_norm': True},
    'clip01': {'clip': 0.1},
}

if RUN_ABLATION_PPO:
    for name, cfg in ablation_ppo.items():
        print('Training %s: %s' % (name, cfg))
else:
    print('Using existing results')

### Observation Ablation Results

In [ ]:
display(Image('results/exp/ablation_obs_chart.png'))

Frame stack 4->1 drops 279 points, 4->2 drops 887. Resolution 80->64 drops 694. Four stacked frames provide important motion information.


Training code:

In [ ]:
RUN_ABLATION_OBS = False

ablation_obs = {
    'stack1': {'frame_stack': 1},
    'stack2': {'frame_stack': 2},
    'res64':  {'obs_res': 64},
}

if RUN_ABLATION_OBS:
    for name, cfg in ablation_obs.items():
        print('Training %s: %s' % (name, cfg))
else:
    print('Using existing results')

### Reward Ablation Results

In [ ]:
display(Image('results/exp/ablation_rew_chart.png'))

Removing step reward drops 970 points. Removing early-crash penalty drops 1274. Removing both (sparse) still reaches 2995 -- reward shaping helps but is not strictly necessary.


Training code:

In [ ]:
RUN_ABLATION_REW = False

ablation_rew = {
    'nostep':    {'step_reward': 0},
    'nopenalty': {'early_penalty': 0},
    'sparse':    {'step_reward': 0, 'early_penalty': 0},
}

if RUN_ABLATION_REW:
    for name, cfg in ablation_rew.items():
        print('Training %s: %s' % (name, cfg))
else:
    print('Using existing results')

### Conclusions

1. **PPO is the only successful vision-based RL method** (2995 perfect, 50/50 episodes at the cap)
2. **GAE is the single most critical component** (removing it drops 2995 -> 744, delta -2175)
3. **Entropy bonus is important** (drops to 1633, delta -1286)
4. **Clip range matters significantly** (dropping to 0.1 gives 1716, delta -1203)
5. **Frame stacking contributes** (4->1 drops 279, 4->2 drops 887)
6. **Reward shaping components matter** (step reward -970, early penalty -1274; sparse still works at 2995)
7. **Vision off-policy methods all fail** (DQN/DDQN/C51 greedy < 66, worse than random)
8. **Obstacle type features matter for state-DQN** (removing them drops 595 points)
9. Advantage normalization has the smallest effect (delta -211)


**Final demo:**

In [ ]:
display(Image('results/dino_score_2191.gif'))

### Cleanup

In [ ]:
if 'asset_server' in dir() and asset_server:
    asset_server.shutdown()
    asset_server.server_close()
    print('Server stopped')